# 01 - Data Loading & Validation

**Unseen-Customer Fraud Detection Using Fraud Probability and Behavioral Novelty**

This notebook loads the synthetic transaction stream, runs the tolerant
schema mapping (`src/data_loader.py`) and produces the structured data-quality
report (`src/data_validation.py`).


In [ ]:
import os, sys
ROOT = os.path.dirname(os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


In [ ]:
from src.config import get_settings
from src.data_loader import load_dataframe, valid_numeric, SchemaMap
from src.data_validation import build_validation_report

cfg = get_settings()
df, schema = load_dataframe(cfg)
df = valid_numeric(df)
print(f"loaded: {len(df):,} rows  x  {df.shape[1]} columns")
print("fraud rate: {:.2f}%".format(df['is_fraud'].mean() * 100))


In [ ]:
rep = build_validation_report(df)
for k in ("rows", "fraud_count", "fraud_percent", "unique_counts",
          "timestamp_range"):
    print(k, "=>", rep[k])


In [ ]:
print("mapped columns (alias -> canonical):")
for canon, raw in (d := schema.mapping).items():
    print(f"  {canon:28s} <- {raw}")
print("unmapped optional:", schema.unmapped)


## Key takeaways
- The loader maps aliases and coerces types; timestamps are parsed and the
  frame sorted chronologically (`ts_sec` attached).
- Duplicate `transaction_id` rows are dropped at load and their count kept.
- Nothing is split or featurised yet – the stream stays untouched until the
  evaluation splits are constructed in notebook 03.
